# Frameworks

Aun siendo una disciplina nueva, ya nos hemos encontrado con cuestiones que plantean una evolución de los agentes y los sistemas productivos. De ahí que empiecen a surgir nuevas formas de interacción y estrategias de construcción de agentes.

### Parlant

[Parlant](https://github.com/emcie-co/parlant) es un framework orientado a corregir un aspecto relevante... Nosotros dedicamos tiempo y horas a ajustar los modelos y los prompts de este, pero cuando el cliente final interactúa:

* Aparecen casos no contemplados
* El agente alucina en momentos críticos
* No presenta buenos resultados en casos no contemplados
* Cada conversación parece poder acabar de forma no esperada

La idea de Parlant es equipar al agente con guías de comportamiento y respuestas enlatadas que permitan controlar situaciones extremas. También incorporar explicabilidad para poder analizar en detalle los casos y presentar mejoras continuas.

[Ejemplos](https://www.parlant.io/docs/quickstart/examples/)

### DSPy

[DSPy](https://dspy.ai/) aboga por desarrollos declarativos. Dejar de enfocarnos en los prompts y enfocarse en la tarea que requerimos que el agente resuelva.

- Charla sobre el enfoque: https://www.youtube.com/watch?v=JEMYuzrKLUw).

DSPy (Declarative Self-improving Python) te permite construir software de IA a partir de módulos en lenguaje natural y componerlos de forma genérica con diferentes modelos, estrategias de inferencia o algoritmos de aprendizaje, en lugar de tener que lidiar con prompts o trabajos de entrenamiento. Esto hace que el software de IA sea más confiable, fácil de mantener y portable entre diferentes modelos y estrategias.

DSPy se integra con MLFlow o Opik para poder trazar nuestros experimentos.

In [1]:
import opik

opik.configure(use_local=False)

OPIK: Your Opik API key is available in your account settings, can be found at https://www.comet.com/api/my/settings/ for Opik cloud
OPIK: Configuration saved to file: /home/iraitz/.opik.config


In [3]:
import os
import dspy
from opik.integrations.dspy.callback import OpikCallback
from dotenv import load_dotenv

load_dotenv(override=True)

#lm = dspy.LM("gemini/gemini-2.5-flash", api_key=os.getenv("GOOGLE_API_KEY"))
lm = dspy.LM('openai/gpt-4o-mini', api_key=os.getenv("OPENAI_API_KEY"))

opik_callback = OpikCallback(project_name="orenes", log_graph=True)
dspy.configure(lm=lm, callbacks=[opik_callback])

Los módulos nos ayudan a describir nuestras interacciones con las LLMs como **módulos**.

In [12]:
math = dspy.ChainOfThought("question -> answer: float") # Solo indicamos el proceso
math(question="Lanzamos dos dados. ¿Cual es la probabilidad de que sumen 2?")

Prediction(
    reasoning='Para que la suma de dos dados sea 2, solo hay una combinación posible: (1, 1). Cada dado tiene 6 caras, por lo que el número total de combinaciones posibles al lanzar dos dados es 6 * 6 = 36. La probabilidad de que la suma sea 2 es el número de combinaciones favorables (1) dividido por el número total de combinaciones (36). Por lo tanto, la probabilidad es 1/36.',
    answer=0.027777777777777776
)

In [13]:
from typing import Literal

class Sentiment(dspy.Signature):
    """Clasifica un texto en el sentimiento dominante"""

    sentence: str = dspy.InputField()
    sentiment: Literal["positivo", "negativo", "neutro"] = dspy.OutputField()
    confidence: float = dspy.OutputField()

classify = dspy.Predict(Sentiment)
classify(sentence="Mejor que la semana pasada, pero no creo que este proyecto me esté ayudando a progresar en mi carrera.")

Prediction(
    sentiment='neutro',
    confidence=0.75
)

También dispone de **optimizadores** y **evaluadores** que nos permiten iterar y mejorar las consultas que resultan. Para ello, como siempre, deberemos tener preparado nuestro set de datos... o emplear un conjunto _benchamrk_ como https://hotpotqa.github.io/

In [4]:
import dspy
from dspy.datasets import HotPotQA

def search_wikipedia(query: str) -> list[str]:
    results = dspy.ColBERTv2(url="http://20.102.90.50:2017/wiki17_abstracts")(query)
    return [x["text"] for x in results]

trainset = [x.with_inputs('question') for x in HotPotQA(train_seed=2024, train_size=10).train]
react = dspy.ReAct("question -> answer", tools=[search_wikipedia])

# Evaluador
tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=2)
optimized_react = tp.compile(react, trainset=trainset)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
2025/09/09 09:39:23 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 8

2025/09/09 09:39:23 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/09/09 09:39:23 INFO dspy.teleprompt.mipro_optimizer_v2: These will be

Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 2/2 [00:08<00:00,  4.12s/it]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 4/6


100%|██████████| 2/2 [00:00<00:00, 12.85it/s]


...

2025/09/09 09:57:15 INFO dspy.evaluate.evaluate: Average Metric: 2 / 8 (25.0%)
2025/09/09 09:57:15 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 25.0 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 5', 'Predictor 1: Instruction 0', 'Predictor 1: Few-Shot Set 1'].
2025/09/09 09:57:15 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [0.0, 0.0, 12.5, 0.0, 0.0, 0.0, 12.5, 0.0, 12.5, 12.5, 12.5, 12.5, 25.0, 25.0, 25.0, 12.5, 25.0, 25.0, 25.0]
2025/09/09 09:57:15 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 25.0
2025/09/09 09:57:15 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/09/09 09:57:15 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 20 / 20 =====



Average Metric: 0.00 / 8 (0.0%): 100%|██████████| 8/8 [01:03<00:00,  7.91s/it]

2025/09/09 09:58:19 INFO dspy.evaluate.evaluate: Average Metric: 0 / 8 (0.0%)
2025/09/09 09:58:19 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 0.0 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 5', 'Predictor 1: Instruction 1', 'Predictor 1: Few-Shot Set 0'].
2025/09/09 09:58:19 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [0.0, 0.0, 12.5, 0.0, 0.0, 0.0, 12.5, 0.0, 12.5, 12.5, 12.5, 12.5, 25.0, 25.0, 25.0, 12.5, 25.0, 25.0, 25.0, 0.0]
2025/09/09 09:58:19 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 25.0
2025/09/09 09:58:19 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/09/09 09:58:19 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 21 / 20 =====



Average Metric: 1.00 / 8 (12.5%): 100%|██████████| 8/8 [01:31<00:00, 11.40s/it]

2025/09/09 09:59:50 INFO dspy.evaluate.evaluate: Average Metric: 1 / 8 (12.5%)
2025/09/09 09:59:50 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 12.5 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 1', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 3'].
2025/09/09 09:59:50 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [0.0, 0.0, 12.5, 0.0, 0.0, 0.0, 12.5, 0.0, 12.5, 12.5, 12.5, 12.5, 25.0, 25.0, 25.0, 12.5, 25.0, 25.0, 25.0, 0.0, 12.5]
2025/09/09 09:59:50 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 25.0
2025/09/09 09:59:50 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/09/09 09:59:50 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 25.0!


In [5]:
optimized_react

react = Predict(StringSignature(question, trajectory -> next_thought, next_tool_name, next_tool_args
    instructions="You are an inquisitive film historian tasked with answering specific questions related to films and their production details. Your goal is to utilize available tools to gather pertinent information that will help you provide accurate answers. Given the fields `question`, produce the fields `answer`. \n\nIn each episode, you will be given the fields `question` as input and can refer to your past trajectory. Interleave `next_thought`, `next_tool_name`, and `next_tool_args` in each turn, and when finishing the task. After each tool call, observe the resulting information, which will be added to your trajectory. \n\nWhen writing `next_thought`, reason about the current situation and plan next steps. For selecting the `next_tool_name`, choose between:\n\n(1) `search_wikipedia`, taking arguments {'query': {'type': 'string'}}.\n(2) `finish`, which indicates that you have all 

Sobre MIPROv2: https://dspy.ai/api/optimizers/MIPROv2/#how-miprov2-works

Más en... https://dspy.ai/tutorials/classification_finetuning/

### AdalFlow

[AdalFlow](https://adalflow.sylph.ai/) extiende las técnicas de optimización y refuerzo a agentes autónomos auto-ajustables. Esto podemos extenderlo a una arquitectura RAG de forma que 

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SylphAI-Inc/AdalFlow/blob/main/notebooks/tutorials/adalflow_rag_optimization.ipynb)

Cada vez surgen más alternativas planteadas llevar al extremo de la autonomía estos agentes: [AutoAgent](https://github.com/HKUDS/AutoAgent)

# Rendimiento

Una disciplina en constante evolución que ofrece mejoras sobre los procesos de inferencia cuando usamos LLMs. Deberemos conocer bien las arquitecturas disponibles en nuestros sistemas ([GPU](https://modal.com/gpu-glossary) sobre todo) para poder exprimir al máximo el uso de los recursos disponibles.

[LMCache](https://lmcache.ai/) es un producto muy específico destinado a ser combinado con herramientas como vLLM. Se centra en el almacenamiento y reusabilidad de las matrices KV generadas durante los procesos de inferencia.

![](https://github.com/LMCache/demo/blob/master/demo4-compare-with-vllm/imgs/demo_4.png?raw=true)

Además nos facilita una calculadora para ver qué impacto esperado podemos esperar: https://lmcache.ai/kv_cache_calculator.html. Siempre bajo las limitaciones de los modelos y hardware que vayamos a ocupar: https://blog.ovhcloud.com/gpu-for-llm-inferencing-guide/

# Datos no estructurados

Gestión de datos no estructurados en nuestros entornos informacionales. 

### Tensorlake

[Tensorlake](https://www.tensorlake.ai/) trabaja a nivel de los pilares que habilitan todos estos procesos y fórmulas: los datos. Funciona en modo API con lo que es necesario tener una cuenta en su plataforma.

### Landing.ai

De la mano de Andrew Ng, [Landing AI](https://landing.ai/) nos ofrece funcionalidades de procesado de documentos. También con modalidad API, podremos iniciarnos de forma gratuita y explorar la solución.

# Guardarailes programáticos

Algunos de los frameworks arriba [ya incluyen](https://www.parlant.io/docs/advanced/custom-llms#moderation-services) ciertos aspectos de control, pero vemos que empiezan a aparecer piezas específicas de software para esta tarea.

### Nemo

![](https://github.com/NVIDIA/NeMo-Guardrails/raw/develop/docs/_static/images/programmable_guardrails.png)

Con foco en la seguridad y los guardarailes (https://github.com/NVIDIA/NeMo-Guardrails) nos permite hacer una implementación sencilla que se haga cargo de validar entradas y salidas en las interacciones de los agentes.

```py
from nemoguardrails import LLMRails, RailsConfig

# Load a guardrails configuration from the specified path.
config = RailsConfig.from_path("PATH/TO/CONFIG")
rails = LLMRails(config)

completion = rails.generate(
    messages=[{"role": "user", "content": "Hello world!"}]
)
```